In [1]:
import os
import sys
from pathlib import Path

import geopandas as gpd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from dotenv import load_dotenv

sys.path.insert(0, '../..')
REPO_ROOT = Path('../..').resolve()
load_dotenv(REPO_ROOT / '.env')

from lvt.lvt_utils import (
    model_split_rate_tax,
    calculate_current_tax,
    calculate_category_tax_summary,
    print_category_tax_summary,
    save_standard_export,
)
from lvt.census_utils import get_census_data_with_boundaries, match_to_census_blockgroups
from lvt.philadelphia import (
    tax_year_params, parcel_cache_path, split_zero_building_parcels, expand_abatement_cohort,
    compute_lycd_land_values, compute_residual_building_value, carry_forward_exemptions,
)

CITY_NAME = 'philadelphia'
STATE_FIPS = '42'
COUNTY_FIPS = '101'
LAND_IMPROVEMENT_RATIO = 4.0

# GMA zone assignment: static reference extracted from OPA's 2025 GMA PDF
# (parcel centroid → L1/L2/L3 zone labels; 17 / 84 / 613 zones)
GMA_PATH = Path('data/parcel_gma_assignment.parquet')

# --- Tax year -------------------------------------------------------------------
# Rates and the revenue-validation target live in lvt/philadelphia.py, keyed by year and
# cited there. Do NOT hardcode a millage here: the combined rate has been 1.3998% for
# years, but the City/School split moved at TY2025, which silently invalidates the
# city-only cross-check without changing anything the model computes.
TAX_YEAR = int(os.environ.get('LVT_TAX_YEAR', 2026))   # override: LVT_TAX_YEAR=2027
TY = tax_year_params(TAX_YEAR)
MILLAGE = TY.combined_mills
PARCEL_PATH = parcel_cache_path(TAX_YEAR)
MODEL_TYPE = f'split_rate_4to1_lycd_post_abatement_ty{TAX_YEAR}'
EXPORT_SUFFIX = f'_lycd_post_abatement_ty{TAX_YEAR}'

print(TY.describe())

DATA_DIR = Path('data')
DATA_DIR.mkdir(exist_ok=True)


TY2026: 0.6159% city + 0.7839% school = 1.3998% (13.998 mills) | city target $891,102,000 (projection) | homestead $100,000


C:\Users\druss\miniconda3\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.4.3)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(


## Step 1: Load parcel data

In [2]:
if not PARCEL_PATH.exists():
    raise FileNotFoundError(
        f'{PARCEL_PATH} not found. Build it with:\n'
        f'    python scripts/build_philadelphia_parcel_cache.py --year {TAX_YEAR}\n'
        'The cache is keyed by tax year on purpose — opa_properties_public always carries '
        'the latest assessment year, so an unsuffixed cache makes it easy to model one '
        "year's taxable values against another year's expectations with no visible symptom."
    )
gdf = gpd.read_parquet(PARCEL_PATH)
_required = {'parcel_number', 'taxable_land', 'taxable_building', 'market_value',
             'exempt_land', 'exempt_building', 'pin', 'category_code', 'total_area'}
_missing = _required - set(gdf.columns)
if _missing:
    raise ValueError(
        f'{PARCEL_PATH} is missing columns {sorted(_missing)} — rebuild with '
        f'scripts/build_philadelphia_parcel_cache.py --year {TAX_YEAR} --force'
    )
gdf['parcel_number'] = gdf['parcel_number'].astype(str).str.zfill(9)
print(f'Loaded {len(gdf):,} parcels for TY{TAX_YEAR}')
print(f'  taxable base: ${(gdf["taxable_land"].sum() + gdf["taxable_building"].sum())/1e9:.3f}B')


Loaded 583,249 parcels for TY2026
  taxable base: $152.997B


## Step 2: Compute LYCD land values (shared construction)

Land value: GMA-hierarchical LYCD, `zone_psf x land_pct x lot_area`, built by
`lvt.philadelphia.compute_lycd_land_values` — the single implementation every
Philadelphia LYCD notebook now shares (`model_lycd.ipynb`, this notebook, and
`model_lycd_reassessment.ipynb`), so the land surface cannot silently diverge between
them. See the function's docstring for the exact algorithm; in summary:

1. **Lot area, one convention, no spatial join.** OPA `total_area` first (~94.5% of
   parcels), then a Mercator-corrected DOR PIN-polygon area, then KNN from parcels
   already on that convention. The method is exactly scale-invariant in area — a
   uniform area error cancels completely — so only a *mixed*-convention error moves
   results. An earlier version of this notebook mixed a true-ground-area source with a
   Web-Mercator-inflated one, tripling the city's apparent land area; the function
   guards against that by raising if the total exceeds 1.5x Philadelphia's actual land
   area.
2. **Zone rate.** `median(market_value / lot_area)` over *improved* parcels, at the
   finest OPA Geographic Market Area level with at least 50 improved parcels
   (L3, else L2, else L1, of 613/84/17 zones).
3. **Land allocation.** 20% of the zone rate for improved parcels (OPA's own default),
   100% for vacant ones — OPA under-assesses bare land, and the allocation preserves
   that development-potential signal rather than suppressing it.
4. **KNN fallback** for the parcels outside GMA coverage, imputing a neighbour's
   *dollar* land value directly rather than a zone rate applied to the parcel's own
   area — a known limitation (audit 2026-08-08 finding 5).
5. **Market-value cap**, improved parcels only: land value cannot exceed the parcel's
   own market value. Vacant parcels are exempt from the cap for the same reason as (3).

`docs/LYCD_LAND_MODEL_ROADMAP.md` records what this method is (the *allocation*
technique, least preferred in the IAAO hierarchy), why each of these choices was made,
and what an assessor-grade replacement would need.

In [3]:
PIN_AREA_PATH = DATA_DIR / 'parcel_areas_by_pin_current.parquet'

if not PIN_AREA_PATH.exists():
    raise FileNotFoundError(
        f'{PIN_AREA_PATH} not found. Build it with:\n'
        '    python scripts/fetch_dor_parcel_areas.py\n'
        'Do NOT fall back to parcel_areas_by_pin.parquet — those areas are Web Mercator '
        '(inflated ~1.704x at this latitude) and mixing them with OPA total_area puts ~5% of '
        'parcels on a different area scale. See the Step 2 notes above.'
    )

pin_areas = pd.read_parquet(PIN_AREA_PATH)
pin_areas['pin'] = pin_areas['pin'].astype(str).str.strip()
print(f'PIN-area lookup: {len(pin_areas):,} PINs (true ground sqft, Mercator-corrected)')
print(f'  median lot: {pin_areas["pin_area_sqft"].median():,.0f} sqft')

PIN-area lookup: 580,097 PINs (true ground sqft, Mercator-corrected)
  median lot: 1,365 sqft


In [4]:
if not GMA_PATH.exists():
    raise FileNotFoundError(
        f'{GMA_PATH} not found. This is the static GMA zone-assignment reference file; '
        'see model_lycd_reassessment.ipynb Step 2 or CLAUDE.md for how it was built.'
    )
gma = pd.read_parquet(GMA_PATH)

lycd = compute_lycd_land_values(gdf, gma, pin_areas)
gdf = lycd.gdf
print(lycd.describe())
print(f"  lot area KNN-imputed: {lycd.diagnostics['n_area_knn']:,} parcels")
print(f"  land value KNN-imputed (no GMA zone): {lycd.diagnostics['n_land_knn']:,} parcels "
      "-- a spatial smooth of a neighbour's dollar land value, not this parcel's own zone "
      "rate x area; see docs/LYCD_LAND_MODEL_ROADMAP.md.")

Lot area source: opa_total_area=550,807, knn=30,696, pin_dor=1,351, pin_override=395
  OPA records overridden by surveyed polygon: 395
  total lot area = 0.84x the city (expect <1.0)
GMA assignment: 527,364 matched (90.4%); L3=526,329, knn=55,885, L2=929, L1=106
Market-value cap (improved only): 15,215 parcels, $130.95B -> $108.40B (17.2% removed)
  lot area KNN-imputed: 30,696 parcels
  land value KNN-imputed (no GMA zone): 55,885 parcels -- a spatial smooth of a neighbour's dollar land value, not this parcel's own zone rate x area; see docs/LYCD_LAND_MODEL_ROADMAP.md.


## Step 3: Summarize LYCD land base

In [5]:
total_lycd_land = gdf['lycd_land_value'].sum()
total_opa_land  = gdf['taxable_land'].sum()
print(f'Total LYCD land base:    ${total_lycd_land/1e9:.2f}B')
print(f'Total OPA taxable land:  ${total_opa_land/1e9:.2f}B')
print(f'Ratio LYCD/OPA:          {total_lycd_land/total_opa_land:.2f}x')
print()
print('LYCD land value by GMA level (median $):')
print(gdf.groupby('gma_level')['lycd_land_value'].median().sort_values(ascending=False).to_string())
print()
print('LYCD land value percentiles (all parcels):')
for p in [10, 25, 50, 75, 90, 99]:
    v = gdf['lycd_land_value'].quantile(p/100)
    print(f'  p{p:2d}: ${v:,.0f}')


Total LYCD land base:    $108.40B
Total OPA taxable land:  $43.00B
Ratio LYCD/OPA:          2.52x

LYCD land value by GMA level (median $):
gma_level
L2     1.045000e+06
L1     6.270000e+05
knn    6.068347e+04
L3     4.593303e+04

LYCD land value percentiles (all parcels):
  p10: $19,022
  p25: $29,183
  p50: $46,654
  p75: $79,393
  p90: $159,600
  p99: $1,426,894


## Step 4: Categorize parcels (same overrides as OPA model)

In [6]:
gdf['category_code'] = (
    pd.to_numeric(gdf['category_code'], errors='coerce')
    .astype('Int64')
    .astype(str)
)

CATEGORY_MAP = {
    '1':  'Single Family Residential',
    '2':  'Small Multi-Family (2-4 units)',
    '3':  'Mixed Use',
    '4':  'Commercial',
    '5':  'Industrial',
    '6':  'Vacant Land',
    '7':  'Other Commercial',
    '8':  'Other Residential',
    '9':  'Hotel',
    '10': 'Office / Commercial Condo',
    '11': 'Other',
    '12': 'Vacant Land',
    '13': 'Vacant Land',
    '14': 'Large Multi-Family (5+ units)',
    '15': 'Retail / General Commercial',
}
gdf['PROPERTY_CATEGORY'] = gdf['category_code'].map(CATEGORY_MAP).fillna('Other')

# Override 1: $0 improvement -> Vacant Land
gdf.loc[gdf['taxable_building'] <= 0, 'PROPERTY_CATEGORY'] = 'Vacant Land'

# Override 2: a $0 taxable building line has three different causes, and calling all of
# them "abated" put ~13K homesteaded rowhomes in the abated bucket -- then revoked their
# Homestead Exemption under the reform. Split on the year's statutory homestead cap.
GENUINE_VACANT_CODES = {'6', '12', '13'}
_zb = split_zero_building_parcels(
    gdf, gdf['PROPERTY_CATEGORY'], TY.homestead_exemption, CATEGORY_MAP,
    genuine_vacant_codes=tuple(GENUINE_VACANT_CODES),
)
gdf['PROPERTY_CATEGORY'] = _zb.category
abated_mask = _zb.abated
print(_zb.describe())

# Override 3: OPA-vacant with nonzero building value
improved_vacant_mask = (
    gdf['category_code'].isin(GENUINE_VACANT_CODES) &
    (gdf['taxable_building'] > 0)
)
gdf.loc[improved_vacant_mask, 'PROPERTY_CATEGORY'] = 'Improved Vacant Land'

# Override 3.5: Override 2's zero-building test only catches a FULL (100%-exempt)
# construction abatement. A graduated residential abatement past year 1, the 90%
# commercial/industrial schedule, and a rehab abatement all leave a taxable building line,
# so that test cannot see them — they were left in their ordinary category, understating
# the post-abatement baseline and, further down, letting their exemption be silently
# carried forward by carry_forward_exemptions as if it were unrelated relief (that function
# excludes `abated` rows entirely; missed rows fell through into `ordinary_taxable`). Read
# the full population from scripts/build_philadelphia_abatement_classification.py's
# billed-history classification and reclassify anything it calls an abatement that
# Override 2 did not already catch. See cities/philadelphia/CLAUDE.md, "The zero-building
# test finds only full abatements".
exp = expand_abatement_cohort(gdf, gdf['PROPERTY_CATEGORY'], abated_mask, TAX_YEAR, data_dir=DATA_DIR)
gdf['PROPERTY_CATEGORY'] = exp.category
abated_mask = exp.abated
print(exp.describe())

gdf['taxable_total'] = (gdf['taxable_land'] + gdf['taxable_building']).clip(lower=0)
gdf['full_exmp'] = (gdf['taxable_total'] <= 0).astype(int)

# Override 4: fully exempt parcels
EXEMPT_CATEGORY_MAP = {k: v + ' — Exempt' for k, v in CATEGORY_MAP.items()}
exempt_mask = gdf['full_exmp'] == 1
gdf.loc[exempt_mask, 'PROPERTY_CATEGORY'] = (
    gdf.loc[exempt_mask, 'category_code']
    .map(EXEMPT_CATEGORY_MAP)
    .fillna('Other — Exempt')
)

print(f'Total parcels: {len(gdf):,}')
print(f'Fully exempt: {gdf["full_exmp"].sum():,}  |  '
      f'Abated: {abated_mask.sum():,}  |  '
      f'Improved vacant: {improved_vacant_mask.sum():,}  |  '
      f'Taxable: {(gdf["full_exmp"] == 0).sum():,}')
print()
print('Property category distribution:')
print(gdf['PROPERTY_CATEGORY'].value_counts().to_string())

zero-building line: 14,287 abated | 13,995 homestead-zeroed (96.3% confirmed by OPA's homestead flag) | 1,119 genuinely $0 improvement


abatement cohort: 23,109 total (8,822 added by the schedule classification, of 20,492 the script classifies as a construction abatement) | restored building value: $23.59B
Total parcels: 583,249
Fully exempt: 36,932  |  Abated: 23,109  |  Improved vacant: 880  |  Taxable: 546,317

Property category distribution:
PROPERTY_CATEGORY
Single Family Residential                  423750
Small Multi-Family (2-4 units)              37570
Vacant Land                                 30548
Abated / Construction Exemption             23109
Single Family Residential — Exempt          20078
Mixed Use                                   13519
Vacant Land — Exempt                        11714
Commercial                                   8714
Industrial                                   3516
Commercial — Exempt                          3298
Large Multi-Family (5+ units)                2533
Other Residential                            1089
Small Multi-Family (2-4 units) — Exempt      1026
Improved Vacant La

## Step 5: Post-Abatement Baseline Tax

Pre-shift baseline treats all active construction abatements as **expired**. The abated cohort
is the union of two tests: parcels whose taxable building line nets to $0 under an active
exemption (`split_zero_building_parcels`, a full 100%-exempt abatement), and parcels
`scripts/build_philadelphia_abatement_classification.py` identifies from their billed exemption
history as a graduated residential, 90% commercial, or rehab abatement — schedules that still
carry a taxable building line and so are invisible to the zero-building test alone
(`expand_abatement_cohort`). For every abated parcel the gross (pre-exemption) building value
— `taxable_building + exempt_building` — is restored to the taxable base before computing
`current_tax`. This produces a **higher revenue baseline than FY2024 actuals** — the LVT reform
is revenue-neutral relative to this higher counterfactual baseline.


In [7]:
gdf['millage_rate'] = MILLAGE

# Post-abatement baseline: restore abated building values.
gdf['model_building'] = pd.to_numeric(gdf['taxable_building'], errors='coerce').fillna(0).clip(lower=0)

abated = gdf['PROPERTY_CATEGORY'] == 'Abated / Construction Exemption'
# Guard: PROPERTY_CATEGORY must still agree with expand_abatement_cohort's own mask -- if
# something reset the category between there and here, the two would silently diverge and
# this cell would restore the wrong set of parcels.
assert (abated == exp.abated).all(), (
    "PROPERTY_CATEGORY == 'Abated / Construction Exemption' no longer matches the expanded "
    'abatement cohort from expand_abatement_cohort -- something changed PROPERTY_CATEGORY '
    'in between.'
)
gdf.loc[abated, 'model_building'] = exp.restored_building[abated].values

n_zero_building = int((abated & (gdf['taxable_building'] <= 0)).sum())
n_still_taxable  = int((abated & (gdf['taxable_building'] > 0)).sum())
abated_bldg_total = gdf.loc[abated, 'model_building'].sum()
print(f'Abated parcels, zero-building test caught:      {n_zero_building:,}')
print(f'Abated parcels, schedule classification added:  {n_still_taxable:,} '
      f'(still carried a taxable building line -- graduated/90%/rehab schedules)')
print(f'Total abated building base restored:            ${abated_bldg_total/1e9:.2f}B')
print()

# post_abatement_total: OPA taxable_land + model_building (abatements treated as expired)
gdf['post_abatement_total'] = (
    pd.to_numeric(gdf['taxable_land'], errors='coerce').fillna(0).clip(lower=0)
    + gdf['model_building']
).clip(lower=0)

current_revenue, _, gdf = calculate_current_tax(
    df=gdf,
    tax_value_col='post_abatement_total',
    millage_rate_col='millage_rate',
    exemption_flag_col='full_exmp',
)

pre_abatement_levy = gdf['taxable_total'].mul(MILLAGE / 1000).sum()
print(f'Post-abatement baseline (city + school): ${current_revenue:,.0f}')
print(f'Pre-abatement combined levy (modeled):   ${pre_abatement_levy:,.0f}')
print(f'Added by restoring abated buildings:     ${current_revenue - pre_abatement_levy:,.0f}')


Abated parcels, zero-building test caught:      14,781
Abated parcels, schedule classification added:  8,328 (still carried a taxable building line -- graduated/90%/rehab schedules)
Total abated building base restored:            $23.59B



Post-abatement baseline (city + school): $2,440,388,629
Pre-abatement combined levy (modeled):   $2,141,653,043
Added by restoring abated buildings:     $298,735,587


## Step 6: Build LYCD Reform Base (Post-Abatement)

Land value: GMA hierarchical LYCD (`lycd_land_value`), never adjusted downward — see below.
Revenue target: post-abatement baseline `current_tax` from Step 5.

**Abated parcels** (the expanded cohort from Step 4's Override 3.5): `model_building` from
Step 5 (gross-building restoration) is kept exactly as set there. It is not touched by the
residual-building correction below, and it must never reach `carry_forward_exemptions` at
all — routing an abated row's own exempt dollars through that function as an override
cancels them to zero rather than preserving them, since its exemption-netting always
subtracts a parcel's own historical exempt total from whatever building value it is given.
See `compute_residual_building_value`'s docstring for the full account, or
`docs/LYCD_LAND_MODEL_ROADMAP.md` Stage A item 4. This is also why the expanded cohort
matters here specifically, not only in Step 5: a classified-abatement parcel the
zero-building test missed would otherwise fall into `ordinary_taxable` below and have its
still-active exemption carried forward as if it were unrelated relief — silently keeping it
exempt under the reform instead of expiring it. Guarded directly below.

**Ordinary (non-abated) parcels already taxable today**: Step 5 set `model_building` to
plain OPA `taxable_building`, which is fine for the post-abatement *baseline* (it pairs
with OPA's own `taxable_land`, no double-count) but not for *this* reform base, which
pairs it with LYCD's much larger land value instead. Holding building fixed there means
`model_land + model_building` can exceed `market_value` — not an edge case (audit
2026-08-08 finding 2, re-measured 2026-09-05: 24.4% of taxable non-vacant parcels, $9.31B
in aggregate, on the pre-abatement model; the post-abatement reform base is built the same
way). Land is never reduced to fix this — capping it was tried and collapses toward OPA's
own default ratio for most parcels, undoing the correction LYCD exists to make. Instead,
`compute_residual_building_value` computes `market_value - land` as the gross building
figure, and `carry_forward_exemptions`'s `gross_building_override` re-applies OPA's own
exemption rule (homestead, institutional) to that pair so relief nets exactly once.
Restricted to parcels already in today's taxable base (`full_exmp == 0`), same as
`model_lycd.ipynb` — homestead-wiped re-entry is `model_lycd_reassessment.ipynb`'s
question, not this notebook's.

In [8]:
gdf['model_land'] = gdf['lycd_land_value'].clip(lower=0)
# model_building for abated parcels already set in Step 5 (exempt_building restoration) --
# kept as-is, not touched below.
market_val = pd.to_numeric(gdf['market_value'], errors='coerce').fillna(0)

# Ordinary (non-abated) parcels already in today's taxable base: Step 5's model_building
# (plain OPA taxable_building) paired fine with OPA's own taxable_land for the baseline, but
# pairing it with LYCD's much larger land value here can push land + building above
# market_value. Hold land at LYCD's full value and let building absorb the correction
# instead of capping land. Restricted to full_exmp == 0 so today's fully-exempt parcels are
# left untouched (their re-entry is a separate question -- see the Step 6 markdown).
ordinary_taxable = ~abated & (gdf['full_exmp'] == 0)

# Guard: no parcel the schedule classification calls an abatement may reach
# carry_forward_exemptions as "ordinary" -- that function carries a non-abated parcel's
# exempt dollars forward unchanged, which for a still-active abatement would silently keep
# it exempt under the reform instead of expiring it. `abated` already excludes every
# classified-abatement parcel by construction (Override 3.5); this checks that directly
# rather than trusting the construction.
assert not (ordinary_taxable & exp.classified).any(), (
    f'{int((ordinary_taxable & exp.classified).sum()):,} classified-abatement parcels are '
    'in ordinary_taxable -- their exemption would be carried forward by '
    'carry_forward_exemptions as if it were unrelated relief instead of expiring.'
)

residual_bldg = compute_residual_building_value(gdf.loc[ordinary_taxable], land_col='model_land')
cfe = carry_forward_exemptions(
    gdf.loc[ordinary_taxable], new_land_col='model_land',
    homestead_cap=TY.homestead_exemption, gross_building_override=residual_bldg,
)
assert cfe.diagnostics['n_homestead_wiped_reentering'] == 0, (
    'Restricting the carry-forward call to already-taxable parcels should make homestead '
    "re-entry impossible (those parcels aren't in the input) -- something is wrong if this fires."
)
old_ordinary_bldg = gdf.loc[ordinary_taxable, 'model_building'].copy()
gdf.loc[ordinary_taxable, 'model_building'] = cfe.reform_taxable_building.values

n_bldg_changed = int((cfe.reform_taxable_building.values != old_ordinary_bldg.values).sum())
print(f'Ordinary taxable parcels: {int(ordinary_taxable.sum()):,}')
print(f'  building value revised (exemption-aware residual): {n_bldg_changed:,}')
print(f'  exemption carry-forward reconstruction match rate: {cfe.diagnostics["reconstruction_match_rate"]:.2%}')
print(f'  building base before: ${old_ordinary_bldg.sum()/1e9:.2f}B  ->  after: ${gdf.loc[ordinary_taxable, "model_building"].sum()/1e9:.2f}B')

print(f'\nReform land base (LYCD):        ${gdf["model_land"].sum()/1e9:.2f}B')
print(f'Reform improvement base:         ${gdf["model_building"].sum()/1e9:.2f}B')
print(f'  of which abated bldg:          ${gdf.loc[abated, "model_building"].sum()/1e9:.2f}B')
print(f'OPA taxable land base:           ${pd.to_numeric(gdf["taxable_land"], errors="coerce").sum()/1e9:.2f}B')
print(f'OPA taxable building base:       ${pd.to_numeric(gdf["taxable_building"], errors="coerce").sum()/1e9:.2f}B')

# Guard: for the population this fix targets, land + building must not exceed market_value.
# Vacant land is excluded -- it is deliberately uncapped by compute_lycd_land_values's own
# design and is not part of what this mechanism addresses.
overshoot = ((gdf['model_land'] + gdf['model_building']) - market_val) > 1
vacant_code = gdf['category_code'].isin({'6', '12', '13'})
bad = overshoot & ordinary_taxable & ~vacant_code
n_bad = int(bad.sum())
print(f'\nNon-vacant ordinary-taxable parcels where land + building still exceeds market_value: {n_bad:,}')
assert n_bad == 0, (
    f'{n_bad:,} non-vacant ordinary-taxable parcels still have land + building above '
    'market_value after the exemption-aware residual fix -- see '
    'docs/LYCD_LAND_MODEL_ROADMAP.md Stage A item 4.'
)

Ordinary taxable parcels: 523,208
  building value revised (exemption-aware residual): 479,327
  exemption carry-forward reconstruction match rate: 99.60%
  building base before: $107.75B  ->  after: $103.30B

Reform land base (LYCD):        $108.40B
Reform improvement base:         $126.89B
  of which abated bldg:          $23.59B
OPA taxable land base:           $43.00B
OPA taxable building base:       $110.00B

Non-vacant ordinary-taxable parcels where land + building still exceeds market_value: 0


## Step 7: Revenue-neutral split-rate model (4:1 land:improvement)

In [9]:
taxable = gdf[gdf['full_exmp'] == 0].copy()

land_millage, improvement_millage, new_revenue, taxable = model_split_rate_tax(
    df=taxable,
    land_value_col='model_land',
    improvement_value_col='model_building',
    current_revenue=taxable['current_tax'].sum(),
    land_improvement_ratio=LAND_IMPROVEMENT_RATIO,
)

# Recombine exempt parcels
exempt = gdf[gdf['full_exmp'] == 1].copy()
exempt['new_tax'] = 0.0
exempt['tax_change'] = 0.0
exempt['tax_change_pct'] = 0.0
exempt['taxable_land_value'] = 0.0
exempt['taxable_improvement_value'] = 0.0
gdf = pd.concat([taxable, exempt]).sort_index()

print(f'Land millage:        {land_millage:.4f} mills')
print(f'Improvement millage: {improvement_millage:.4f} mills')
print(f'Revenue check:       ${new_revenue:,.0f} (target: ${taxable["current_tax"].sum():,.0f})')
print()

category_summary = calculate_category_tax_summary(
    df=gdf,
    category_col='PROPERTY_CATEGORY',
    current_tax_col='current_tax',
    new_tax_col='new_tax',
)
print_category_tax_summary(category_summary, title='Philadelphia — 4:1 Split-Rate Tax Impact (LYCD Land Values)')

Land millage:        24.9248 mills
Improvement millage: 6.2312 mills
Revenue check:       $2,440,388,629 (target: $2,440,388,629)




Philadelphia — 4:1 Split-Rate Tax Impact (LYCD Land Values)
                               Category  Count Total Tax Δ ($) Total Δ (%) Mean Δ ($) Median Δ ($) Avg % Δ Median % Δ % Parcels > +10% % Parcels < -10%
              Single Family Residential 423750   $-201,853,911      -16.6%      $-476        $-413   15.4%     -19.8%            19.1%            63.8%
         Small Multi-Family (2-4 units)  37570    $-63,579,294      -26.0%    $-1,692      $-1,147  -22.7%     -28.9%             6.2%            85.5%
                            Vacant Land  30548    $458,446,990      876.9%    $15,007       $2,830 1435.5%     642.7%            97.8%             1.0%
        Abated / Construction Exemption  23109   $-151,371,026      -38.9%    $-6,550      $-3,016  -36.2%     -43.3%             4.2%            91.9%
     Single Family Residential — Exempt  20078              $0        0.0%         $0           $0    0.0%       0.0%             0.0%             0.0%
                           

## Step 8: Census join

In [10]:
import concurrent.futures

_fips = STATE_FIPS + COUNTY_FIPS
try:
    with concurrent.futures.ThreadPoolExecutor(max_workers=1) as _ex:
        _future = _ex.submit(get_census_data_with_boundaries, _fips, 2022)
        try:
            census_data, census_gdf = _future.result(timeout=90)
            gdf = match_to_census_blockgroups(gdf, census_gdf)
            if 'minority_pct' not in gdf.columns and 'total_pop' in gdf.columns and 'white_pop' in gdf.columns:
                gdf['minority_pct'] = ((gdf['total_pop'] - gdf['white_pop']) / gdf['total_pop'] * 100).round(2)
            if 'black_pct' not in gdf.columns and 'total_pop' in gdf.columns and 'black_pop' in gdf.columns:
                gdf['black_pct'] = (gdf['black_pop'] / gdf['total_pop'] * 100).round(2)
            print(f'Census join: {gdf["std_geoid"].notna().mean()*100:.1f}% matched')
        except concurrent.futures.TimeoutError:
            print('Census API timed out — skipping census join')
            for _col in ['std_geoid', 'median_income', 'minority_pct', 'black_pct']:
                gdf[_col] = float('nan')
except Exception as e:
    print(f'Census join failed: {e}')
    for _col in ['std_geoid', 'median_income', 'minority_pct', 'black_pct']:
        gdf[_col] = float('nan')

Census join: 100.0% matched


## Step 9: Export and visualize

In [11]:
out_df = save_standard_export(
    df=gdf,
    city=f'{CITY_NAME}{EXPORT_SUFFIX}',
    output_path=f'../../analysis/data/{CITY_NAME}{EXPORT_SUFFIX}.csv',
    model_type=MODEL_TYPE,
    land_millage=land_millage,
    improvement_millage=improvement_millage,
    property_category_col='PROPERTY_CATEGORY',
    current_tax_col='current_tax',
    new_tax_col='new_tax',
    tax_change_col='tax_change',
    tax_change_pct_col='tax_change_pct',
    taxable_land_col='taxable_land_value',
    taxable_improvement_col='taxable_improvement_value',
    parcel_id_col='parcel_number',
)

from lvt.viz import create_city_report
create_city_report(out_df, f'{CITY_NAME}{EXPORT_SUFFIX}', show=False)
print('Done.')

  [warn] philadelphia_lycd_post_abatement_ty2026: non-standard property categories (will be preserved): ['Abated / Construction Exemption', 'Commercial — Exempt', 'Hotel — Exempt', 'Improved Vacant Land', 'Industrial — Exempt', 'Large Multi-Family (5+ units) — Exempt', 'Mixed Use — Exempt', 'Office / Commercial Condo — Exempt', 'Other Commercial — Exempt', 'Other Residential — Exempt', 'Other — Exempt', 'Retail / General Commercial — Exempt', 'Single Family Residential — Exempt', 'Small Multi-Family (2-4 units) — Exempt', 'Vacant Land — Exempt']


  ✓ philadelphia_lycd_post_abatement_ty2026: 583,249 rows → ../../analysis/data/philadelphia_lycd_post_abatement_ty2026.csv  [model: split_rate_4to1_lycd_post_abatement_ty2026]


create_city_report: excluded 36,932 fully-exempt parcels (583,249 → 546,317 modeled).


Done.
